# IOAI — 2026 Local Mock Parkinson (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-parkinson.zip', '/tmp/park.zip')
    with zipfile.ZipFile('/tmp/park.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Parkinson — 파킨슨 위험/진단 (RoAI/ONIA 2026 Mock)

임상 표 데이터로 위험점수 계산과 진단 분류. 서브태스크 3개:
1. (결정적) CardiometabolicRiskScore = Hypertension + Diabetes + (BMI>30)
2. (결정적) LifestyleRiskIndex = Smoking + (AlcoholConsumption>2) + (PhysicalActivity<1)
3. (ML) `Diagnosis` **확률** 예측 — ROC-AUC 로 채점

**제출**: `submission.csv` 롱포맷 `PatientID, subtaskID, Answer` (subtaskID=`Task1/Task2/Task3`).
**채점(로컬)**: train held-out 으로 ST(Task3) ROC-AUC. **데이터**: `data/train.csv`(+`data/test.csv`).
원본: platform.olimpiada-ai.ro/competitions/7

In [ ]:
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier

root_path = "data"
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df = pd.read_csv(f"{root_path}/test.csv")
train_df.head()

In [ ]:
# Task1 (결정적): CardiometabolicRiskScore = Hypertension + Diabetes + (BMI>30)
# Task2 (결정적): LifestyleRiskIndex = Smoking + (AlcoholConsumption>2) + (PhysicalActivity<1)
subtask1 = (test_df["Hypertension"] + test_df["Diabetes"] + (test_df["BMI"]>30)).astype(int)
subtask2 = (test_df["Smoking"] + (test_df["AlcoholConsumption"]>2).astype(int) + (test_df["PhysicalActivity"]<1).astype(int)).astype(int)

In [ ]:
# Task3 (ML): 진단 확률 — 간단 베이스라인(얕은 결정트리). 모범답안=HistGradientBoosting.
# TODO: 결측/범주 처리·부스팅·튜닝으로 개선 (모범답안 Solution/ 참고)
feat = [c for c in train_df.select_dtypes(include=np.number).columns if c not in ("Diagnosis","PatientID")]
mean = train_df[feat].mean()
Xtr, ytr = train_df[feat].fillna(mean), train_df["Diagnosis"]
Xte = test_df[feat].fillna(mean)
clf = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
subtask3 = clf.predict_proba(Xte)[:, 1]

In [ ]:
# 제출(롱포맷): PatientID, subtaskID(Task1/Task2/Task3), Answer
def build_df(sid, ans):
    return pd.DataFrame({"PatientID": test_df["PatientID"], "subtaskID": sid, "Answer": np.asarray(ans)})
submission_df = pd.concat([build_df("Task1", subtask1), build_df("Task2", subtask2), build_df("Task3", subtask3)])
submission_df.to_csv(f"{root_path}/submission.csv", index=False)
submission_df.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)